# Model Evaluation
**COEN 330 - Applied Machine Learning**

Evaluates all five tuned models on the separated test set (20%).  
Models and the train/test split are loaded from `03_model_training` outputs.

According to project guidelines:\
**Section 5.7**: Test-set metrics, threshold tuning, comparison table, confusion matrices.  
**Section 5.8**: Error analysis, feature importance interpretation, model behaviour comparison, limitations.

In [ ]:
import sys, json
import os
from pathlib import Path
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd

from preprocessing import load_data, split_data
from utils import DATA_RAW, DATASET_FILE, MODELS_DIR, RESULTS_DIR, PLOTS_DIR, SEED
from evaluate import (
    MODEL_FILES, SELECTION_METRIC,
    set_plot_style, load_models, load_cv_results, format_cv_results,
    get_scores, get_oof_scores, tune_threshold, tune_all_thresholds,
    evaluate_models, style_results,
    plot_confusion_matrices, plot_pr_curves, plot_importances,
    outcome_masks, outcome_profile,
)

set_plot_style()

DATA_PATH = DATA_RAW / DATASET_FILE


## 1. Reproduce Train / Test Split & Load Models

The split uses the same seed as `03_model_training`, so `X_test` / `y_test` are identical.  
Each loaded pipeline is a full sklearn Pipeline (raw DataFrame → prediction), so no separate preprocessing is needed.

In [ ]:
df = load_data(DATA_PATH)
X_train, X_test, y_train, y_test = split_data(df)

print(f'Train : {len(X_train):,} rows  |  approved (1) = {y_train.mean():.1%}')
print(f'Test  : {len(X_test):,} rows  |  approved (1) = {y_test.mean():.1%}')

In [ ]:
models = load_models()


## 2. CV Results from Training (Reference)

These are the 5-fold cross-validation means from `03_model_training`. They are displayed only for comparison and are **not used to select the primary model** in this notebook. Model selection is based on test-set F1 score in Section 5.

In [ ]:
cv_df = load_cv_results()

print('5-Fold CV Results (from training):')
display(format_cv_results(cv_df))


---
# Section 5.7: Model Evaluation
## 4. Threshold Tuning (Out-of-Fold on Training Set)

Thresholds are chosen using 5-fold out-of-fold predictions on `X_train`/`y_train` but never on `X_test`. So each score comes from a model clone that did not see that row during fitting. This avoids the optimism of tuning on the same in-sample predictions the model was fit on, which would be especially inflated for the tree ensembles.


In [ ]:
best_thresholds = tune_all_thresholds(models, X_train, y_train)


## 5. Test Set Evaluation - All Models

All models are evaluated on `X_test`/`y_test`. The **primary model is selected as the model with the highest test-set F1 score**.

In [ ]:
results_df = evaluate_models(
    models, X_test, y_test, best_thresholds,
    sort_by=SELECTION_METRIC,
    save_path=RESULTS_DIR / 'test_results.csv',
)

print(f'Test Set Results (sorted by {SELECTION_METRIC}):')
display(style_results(results_df))

test_winner = results_df.iloc[0]
print(
    f"\nSelected primary model from the test set: {test_winner['Model']} "
    f"({SELECTION_METRIC}={test_winner[SELECTION_METRIC]:.4f})"
)
print('Saved to results/test_results.csv')


## 6. Confusion Matrices - All Models

In [ ]:
plot_confusion_matrices(
    models, X_test, y_test, best_thresholds,
    results_df=results_df,
    save_path=str(PLOTS_DIR / 'confusion_matrices_all.png'),
)


## 7. PR-AUC Curves - All Models

In [ ]:
plot_pr_curves(
    models, X_test, y_test,
    save_path=str(PLOTS_DIR / 'pr_curves_all.png'),
)


## 8. Model Complexity, Interpretability & Computational Cost

| Model | Interpretability | Complexity | Training Cost | Notes |
|---|---|---|---|---|
| Logistic Regression | High - coefficients directly readable | Low | Fast (seconds) | Linear boundary; struggles with non-linear patterns |
| SVM (RBF) | Low - kernel space not interpretable | Medium | Slow (minutes on 36K rows) | Strong margin-based classifier; no probability output by default |
| Gaussian Naive Bayes | Medium - per-feature likelihoods | Very low | Fastest | Independence assumption violated by OHE features; high recall but low precision |
| Random Forest | Medium - feature importances available | High | Moderate (400 trees) | Robust; averages many trees; less prone to overfitting than single DT |
| Gradient Boosting | Low - ensemble of trees | High | Moderate | Sequential boosting captures complex patterns; selected only if it has the highest test-set F1 |


---
# Section 5.8: Qualitative Analysis & Interpretation
## 9. Error Analysis - Best Model

We focus error analysis on the **primary model selected by the highest test-set F1 score** from Section 5.

This is a loan-**approval** model: `loan_status` reflects the bank's historical decision, not a verified default or repayment outcome. So:
- **False Positives**: predicted Approved, but the historical decision was Rejected. These are rejected applications whose characteristics resemble historically approved applications.
- **False Negatives**: predicted Rejected, but the historical decision was Approved. These are approved applications whose characteristics resemble historically rejected applications.


In [ ]:
# Select the primary model using TEST-SET performance from Section 5.
# results_df is already sorted from highest to lowest F1.
best_row = results_df.iloc[0]
best_model_name = best_row['Model']
best_selection_score = best_row[SELECTION_METRIC]

best_pipeline = models[best_model_name]
best_t        = best_thresholds[best_model_name]

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

scores_test = get_scores(best_pipeline, X_test_reset)
preds_test  = (scores_test >= best_t).astype(int)

masks = outcome_masks(y_test_reset, preds_test)

print(
    f'Primary model (test-set {SELECTION_METRIC}-selected): '
    f'{best_model_name} ({SELECTION_METRIC}={best_selection_score:.4f}, '
    f'threshold={best_t:.4f})'
)
print(f"True Positives  (correct approvals)  : {masks['TP'].sum():,}")
print(f"True Negatives  (correct rejections) : {masks['TN'].sum():,}")
print(f"False Positives (wrong approvals)    : {masks['FP'].sum():,}")
print(f"False Negatives (wrong rejections)   : {masks['FN'].sum():,}")


In [ ]:
# Sample false positives - loans predicted approved but were rejected
print('=== False Positives (predicted Approved, actually Rejected) ===')
display(X_test_reset[masks['FP']].head(5))


In [ ]:
# Sample false negatives - loans predicted rejected but were approved
print('=== False Negatives (predicted Rejected, actually Approved) ===')
display(X_test_reset[masks['FN']].head(5))


### 9.1 False Positive Profile

Compare the average feature values of false positives vs. true negatives (both historically rejected) to understand what makes a rejected applicant resemble a historically-approved profile to the model.


In [ ]:
numeric_cols = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
                'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']

profile_df = outcome_profile(X_test_reset, masks, numeric_cols)
print('Mean feature values by prediction outcome:')
display(profile_df)


## 10. Feature Importance Interpretation

In [ ]:
with open(MODELS_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)


In [ ]:
for name in ['Gradient Boosting', 'Random Forest', 'Logistic Regression']:
    slug = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    plot_importances(
        models[name], name, feature_names,
        X_eval=X_test, y_eval=y_test,   # used only for GB's permutation-importance fallback
        save_path=str(PLOTS_DIR / f'importance_{slug}.png')
    )


## 11. Why Some Models Performed Better Than Others

**Gradient Boosting** likely performed best because it builds trees sequentially: each tree corrects the errors of the previous one. This lets it capture complex, non-linear interactions between features (e.g. the interaction between `credit_score`, `loan_percent_income`, and `previous_loan_defaults_on_file`) that a linear model can't represent as directly. (The permutation-importance results above are consistent with this, but performance alone doesn't prove this is *the* mechanism: regularisation strength and how each model handles the categorical encoding likely also play a role.)

**Random Forest** performed second-best. It averages many independent trees (bagging), which reduces variance compared to a single decision tree. It trails Gradient Boosting because each tree is built independently rather than correctively.

**SVM (RBF)** performed reasonably but was the slowest to train. The RBF kernel maps data to a higher-dimensional space to find a margin, and it is sensitive to feature scaling. Its `decision_function` score is not a calibrated probability, which limits how the score can be *interpreted* (e.g. as "70% likely approved") — it does not prevent threshold tuning, since the threshold is chosen directly on that score (Section 4).

**Logistic Regression** is a linear model. It can only draw a straight decision boundary in the feature space. The loan approval decision is not linearly separable, which is reflected in its lower test-set classification performance. It is, however, the most interpretable: coefficients directly show which features push toward approval.

**Gaussian Naive Bayes** performs poorly in precision despite achieving high recall. This is because it assumes all features are independent, which is violated here. The one-hot encoded categorical variables (e.g. `loan_intent`) are by construction correlated with each other. As a result, the model overestimates the posterior probability of approval.

## 12. Final Comparison Table (for the Report)

In [ ]:
print('Final Test Set Results -> copy into report Table:')
print(results_df.to_string(index=False))